# **Assignment 3**


**Anja Pijak**  
ap72051@student.uni-lj.si


In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso, Ridge
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV


### Lasso and Ridge

#### Data preparation


In [2]:
column_names = [
    "state", "county", "community", "communityname", "fold", "population", "householdsize",
    "racepctblack", "racePctWhite", "racePctAsian", "racePctHisp", "agePct12t21", "agePct12t29",
    "agePct16t24", "agePct65up", "numbUrban", "pctUrban", "medIncome", "pctWWage", "pctWFarmSelf",
    "pctWInvInc", "pctWSocSec", "pctWPubAsst", "pctWRetire", "medFamInc", "perCapInc", "whitePerCap",
    "blackPerCap", "indianPerCap", "AsianPerCap", "OtherPerCap", "HispPerCap", "NumUnderPov", 
    "PctPopUnderPov", "PctLess9thGrade", "PctNotHSGrad", "PctBSorMore", "PctUnemployed", 
    "PctEmploy", "PctEmplManu", "PctEmplProfServ", "PctOccupManu", "PctOccupMgmtProf", 
    "MalePctDivorce", "MalePctNevMarr", "FemalePctDiv", "TotalPctDiv", "PersPerFam", "PctFam2Par", 
    "PctKids2Par", "PctYoungKids2Par", "PctTeen2Par", "PctWorkMomYoungKids", "PctWorkMom", "NumIlleg", 
    "PctIlleg", "NumImmig", "PctImmigRecent", "PctImmigRec5", "PctImmigRec8", "PctImmigRec10", 
    "PctRecentImmig", "PctRecImmig5", "PctRecImmig8", "PctRecImmig10", "PctSpeakEnglOnly", 
    "PctNotSpeakEnglWell", "PctLargHouseFam", "PctLargHouseOccup", "PersPerOccupHous", 
    "PersPerOwnOccHous", "PersPerRentOccHous", "PctPersOwnOccup", "PctPersDenseHous", "PctHousLess3BR", 
    "MedNumBR", "HousVacant", "PctHousOccup", "PctHousOwnOcc", "PctVacantBoarded", "PctVacMore6Mos", 
    "MedYrHousBuilt", "PctHousNoPhone", "PctWOFullPlumb", "OwnOccLowQuart", "OwnOccMedVal", 
    "OwnOccHiQuart", "RentLowQ", "RentMedian", "RentHighQ", "MedRent", "MedRentPctHousInc", 
    "MedOwnCostPctInc", "MedOwnCostPctIncNoMtg", "NumInShelters", "NumStreet", "PctForeignBorn", 
    "PctBornSameState", "PctSameHouse85", "PctSameCity85", "PctSameState85", "LemasSwornFT", 
    "LemasSwFTPerPop", "LemasSwFTFieldOps", "LemasSwFTFieldPerPop", "LemasTotalReq", "LemasTotReqPerPop", 
    "PolicReqPerOffic", "PolicPerPop", "RacialMatchCommPol", "PctPolicWhite", "PctPolicBlack", 
    "PctPolicHisp", "PctPolicAsian", "PctPolicMinor", "OfficAssgnDrugUnits", "NumKindsDrugsSeiz", 
    "PolicAveOTWorked", "LandArea", "PopDens", "PctUsePubTrans", "PolicCars", "PolicOperBudg", 
    "LemasPctPolicOnPatr", "LemasGangUnitDeploy", "LemasPctOfficDrugUn", "PolicBudgPerPop", 
    "ViolentCrimesPerPop"
]


df = pd.read_csv("datasets/communities.data", names = column_names)

df.head()

# drop the first 5 columns (from assignment guidelines)
df = df.iloc[:, 5:]

# droping columns which are mostly na (from communities.names info)
# all of these have 1675 na rows out of (1994)
na_cols = ["LemasSwornFT", 
    "LemasSwFTPerPop", "LemasSwFTFieldOps", "LemasSwFTFieldPerPop", "LemasTotalReq", "LemasTotReqPerPop", 
    "PolicReqPerOffic", "PolicPerPop", "RacialMatchCommPol", "PctPolicWhite", "PctPolicBlack", 
    "PctPolicHisp", "PctPolicAsian", "PctPolicMinor", "OfficAssgnDrugUnits", "NumKindsDrugsSeiz", 
    "PolicAveOTWorked", "PolicCars", "PolicOperBudg", 
    "LemasPctPolicOnPatr", "LemasGangUnitDeploy", "PolicBudgPerPop"]
df = df.drop(columns=na_cols)

# replace ? with na
df.replace('?', np.nan, inplace=True)
# converting all columns in the df to numeric
df = df.apply(pd.to_numeric)

df.head()



,population,householdsize,racepctblack,racePctWhite,racePctAsian,racePctHisp,agePct12t21,agePct12t29,agePct16t24,agePct65up,...,PctForeignBorn,PctBornSameState,PctSameHouse85,PctSameCity85,PctSameState85,LandArea,PopDens,PctUsePubTrans,LemasPctOfficDrugUn,ViolentCrimesPerPop
0,0.19,0.33,0.02,0.90,0.12,0.17,0.34,0.47,0.29,0.32,...,0.12,0.42,0.50,0.51,0.64,0.12,0.26,0.20,0.32,0.20
1,0.00,0.16,0.12,0.74,0.45,0.07,0.26,0.59,0.35,0.27,...,0.21,0.50,0.34,0.60,0.52,0.02,0.12,0.45,0.00,0.67
2,0.00,0.42,0.49,0.56,0.17,0.04,0.39,0.47,0.28,0.32,...,0.14,0.49,0.54,0.67,0.56,0.01,0.21,0.02,0.00,0.43
3,0.04,0.77,1.00,0.08,0.12,0.10,0.51,0.50,0.34,0.21,...,0.19,0.30,0.73,0.64,0.65,0.02,0.39,0.28,0.00,0.12
4,0.01,0.55,0.02,0.95,0.09,0.05,0.38,0.38,0.23,0.36,...,0.11,0.72,0.64,0.61,0.53,0.04,0.09,0.02,0.00,0.03


In [3]:
X = df.loc[:, df.columns != 'ViolentCrimesPerPop'] # attributes
y = df['ViolentCrimesPerPop'] # target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# fill na with average value
X_train.fillna(X_train.mean(), inplace=True)
X_test.fillna(X_train.mean(), inplace=True)

# first test linear regression when all features are there, as a baseline
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
score = mean_squared_error(y_test, y_pred)
print("MSE before selecting features: ",score)


MSE before selecting features:  0.017528376215177402


#### Choosing alpha (regularisation parameter)

I evaluated Lasso and Ridge models with different values of alpha and chose the ones where MSE is smallest.

In [ ]:
alphas = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100]}

# finding best alpha for ridge
model = Ridge()
ridge = GridSearchCV(model, alphas, cv=5, scoring='neg_mean_squared_error')
ridge.fit(X_train, y_train)
best_parameter_ridge = ridge.best_params_['alpha']
best_ridge = ridge.best_estimator_
mse_ridge = mean_squared_error(y_test, best_ridge.predict(X_test))
print(f"Best alpha for Ridge is {best_parameter_ridge} with MSE: {mse_ridge}")

# finding best alpha for lasso
model = Lasso(max_iter=10000)
lasso = GridSearchCV(model, alphas, cv=5, scoring='neg_mean_squared_error')
lasso.fit(X_train, y_train)
best_parameter_lasso = lasso.best_params_['alpha']
best_lasso = lasso.best_estimator_
mse_lasso = mean_squared_error(y_test, best_lasso.predict(X_test))
print(f"Best alpha for Lasso is {best_parameter_lasso} with MSE: {mse_lasso}")


Best alpha for Ridge is 1 with MSE: 0.01752593368283336
Best alpha for Lasso is 0.001 with MSE: 0.01698511411770846


##### Comparing different feature selection methods

In [5]:
# getting features that lasso (with optimal alpha) chose
lasso_coef = best_lasso.coef_
selected_features_lasso = X.columns[(lasso_coef != 0)].tolist()
print(f"Lasso selected {len(selected_features_lasso)} features: {selected_features_lasso}")

# features from previous homework
forward_selection_features = ['PctKids2Par', 'racePctWhite', 'HousVacant', 'pctUrban', 'PctWorkMom', 'MalePctDivorce', 'PctIlleg', 'PctPersDenseHous', 'PctHousOccup', 'agePct12t29', 'LemasPctOfficDrugUn', 'MedOwnCostPctIncNoMtg', 'NumStreet', 'racepctblack', 'OtherPerCap', 'indianPerCap', 'pctWRetire', 'MalePctNevMarr', 'whitePerCap', 'PopDens', 'PctEmploy', 'pctWWage', 'PctPopUnderPov', 'PctEmplManu', 'perCapInc', 'RentLowQ', 'MedRent', 'PctVacMore6Mos', 'RentHighQ', 'LandArea', 'RentMedian', 'PctBSorMore', 'PctHousNoPhone', 'PctOccupManu', 'PctUsePubTrans']  # Replace with actual features
print(f"Forward Feature Selection selected {len(forward_selection_features)} features: {forward_selection_features}")

# comparing 2 lists
common_features_intersection = set(selected_features_lasso).intersection(forward_selection_features) 
print("\nIntersection:", common_features_intersection)
common_features_diff1 = set(selected_features_lasso).difference(forward_selection_features) 
print("What is in Lasso but not in Forward Selection:", common_features_diff1)
common_features_diff2 = set(forward_selection_features).difference(selected_features_lasso) 
print("What is in forward selection but not in Lasso:", common_features_diff2)



Lasso selected 21 features: ['racepctblack', 'racePctWhite', 'pctUrban', 'pctWInvInc', 'AsianPerCap', 'PctEmplManu', 'MalePctDivorce', 'PctKids2Par', 'PctYoungKids2Par', 'PctWorkMom', 'PctIlleg', 'PctRecImmig10', 'PctPersDenseHous', 'HousVacant', 'PctHousOccup', 'PctVacantBoarded', 'MedRentPctHousInc', 'MedOwnCostPctIncNoMtg', 'NumStreet', 'PctForeignBorn', 'LemasPctOfficDrugUn']
Forward Feature Selection selected 35 features: ['PctKids2Par', 'racePctWhite', 'HousVacant', 'pctUrban', 'PctWorkMom', 'MalePctDivorce', 'PctIlleg', 'PctPersDenseHous', 'PctHousOccup', 'agePct12t29', 'LemasPctOfficDrugUn', 'MedOwnCostPctIncNoMtg', 'NumStreet', 'racepctblack', 'OtherPerCap', 'indianPerCap', 'pctWRetire', 'MalePctNevMarr', 'whitePerCap', 'PopDens', 'PctEmploy', 'pctWWage', 'PctPopUnderPov', 'PctEmplManu', 'perCapInc', 'RentLowQ', 'MedRent', 'PctVacMore6Mos', 'RentHighQ', 'LandArea', 'RentMedian', 'PctBSorMore', 'PctHousNoPhone', 'PctOccupManu', 'PctUsePubTrans']

Intersection: {'HousVacant', 'M

Linear regression on the dataset with the chosen features. We can see in the result that features chosen by Forward Feature Selection give better results in term of both, MSE and MAE.

In [6]:
model = LinearRegression()

model.fit(X_train[selected_features_lasso], y_train)

y_pred1 = model.predict(X_test[selected_features_lasso])
mse = mean_squared_error(y_test, y_pred1)
mae = mean_absolute_error(y_test, y_pred1)
print(f"MSE after Lasso selection: MSE {mse} MAE {mae}")

model.fit(X_train[forward_selection_features], y_train)

y_pred1 = model.predict(X_test[forward_selection_features])
mse = mean_squared_error(y_test, y_pred1)
mae = mean_absolute_error(y_test, y_pred1)
print(f"MSE after Forward Feature Selection: MSE {mse}, MAE {mae}")

MSE after Lasso selection: MSE 0.017640642543213614 MAE 0.09515791213620242
MSE after Forward Feature Selection: MSE 0.016469147435801256, MAE 0.09156259616770719


### Gradient Descent and Stochastic Gradient Descent

#### Data preparation

In [7]:
dfw = pd.read_csv('datasets/winequality-white.csv', delimiter=';')

Xw = dfw.loc[:, dfw.columns != 'quality']
yw = dfw['quality']  # Target
dfw.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


In [8]:
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(Xw, yw, test_size=0.3, shuffle=False)
X_train.fillna(X_train.mean(), inplace=True)
X_test.fillna(X_train.mean(), inplace=True)

# scale the features and add bias for sgd and gd to work properly
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_train = np.c_[np.ones(X_train.shape[0]), X_train]
X_test = np.c_[np.ones(X_test.shape[0]), X_test]


#### Ridge Regression with Gradient Descent

In [9]:
class RidgeRegressionGD:
    def __init__(self, learning_rate=0.01, alpha=0.1, max_iter=1000):
        self.learning_rate = learning_rate
        self.alpha = alpha
        self.max_iter = max_iter
        self.theta = None
        

    def _compute_gradient(self, X: np.ndarray, y: np.ndarray):
        """Compute the gradient for Ridge Regression."""
        m = len(y)
        predictions = X.dot(self.theta)
        errors = predictions - y
        return (1 / m) * X.T.dot(errors) + self.alpha * self.theta

    def fit(self, X: np.ndarray, y: np.ndarray):
        """Fit the Ridge Regression model using gradient descent."""
        n_features = X.shape[1]
        self.theta = np.zeros(n_features)  # Initialize coefficients

        for _ in range(self.max_iter):
            gradient = self._compute_gradient(X, y) # compute the gradient at the current coefficients
            update = self.learning_rate * gradient # compute the update value based on the learning rate

            self.theta -= update # update coefficients 
            

    def predict(self, X: np.ndarray):
        return X.dot(self.theta)


Just checking if the implementation works, this time lr = 0.01 

In [10]:
# train the Ridge Regression model
model = RidgeRegressionGD(learning_rate=0.01, alpha=0.1, max_iter=1000)
model.fit(X_train, y_train)

# Predictions
predictions = model.predict(X_test)
print("MSE: ", mean_squared_error(predictions, y_test))

MSE:  0.6598070207731823


#### Ridge Regression with Stochastic Gradient Descent

In [11]:
class RidgeRegressionSGD:
    def __init__(self, learning_rate=0.01, alpha=0.1, max_iter=1000):
        self.learning_rate = learning_rate
        self.alpha = alpha
        self.max_iter = max_iter
        self.theta = None
        

    def _compute_gradient(self, xi, yi):
        """
        Compute the gradient for a single sample 
        """
        prediction = np.dot(xi, self.theta)
        error = prediction - yi
        return xi * error + self.alpha * self.theta

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.theta = np.zeros(n_features)  # Initialize coefficients

        for _ in range(self.max_iter):
            # Iterate over random samples
            for i in np.random.permutation(n_samples):
                xi = X[i]  # the feature vector for the i-th sample
                yi = y[i]  # the target for the i-th sample
                gradient = self._compute_gradient(xi, yi)
                update = self.learning_rate * gradient
                self.theta -= update


    def predict(self, X: np.ndarray):
        return np.dot(X, self.theta)


In [12]:
# Initialize and train the Ridge Regression model using SGD
sgd_model = RidgeRegressionSGD(learning_rate=0.01, alpha=0.1, max_iter=1000)
sgd_model.fit(X_train, y_train)

# Predictions
predictions = sgd_model.predict(X_test)
print("Predicted values:", mean_squared_error(predictions, y_test))

Predicted values: 0.6444848715271245


#### Comparing the two

In [16]:
from tabulate import tabulate
import time

# learning rates proposed in lecture
learning_rates = [0.001, 0.003,0.06, 0.01, 0.03, 0.06, 0.1]

results = [] 
best_lr = None
best_mse = None
time_tracker = {}

print("Ridge GD:")
for lr in learning_rates:
    start = time.time()
    model = RidgeRegressionGD(learning_rate=lr, alpha=0.1, max_iter=1000)
    rgd = model.fit(X_train, y_train)
    y_predicted = model.predict(X_test)
    mse = mean_squared_error(y_test, y_predicted)

    duration = time.time() - start
    time_tracker[lr] = duration
    
    results.append({"Learning Rate": lr, "MSE": mse, "Time (s)": duration})
    
    if best_lr is None or mse < best_mse:
        best_lr = lr
        best_mse = mse

# results as a table
print(tabulate(results, headers="keys", tablefmt="grid"))

print(f"Optimal learning rate: {best_lr} with MSE: {best_mse} executed in time: {time_tracker[best_lr]}s")



Ridge GD:
+-----------------+----------+------------+
|   Learning Rate |      MSE |   Time (s) |
+=================+==========+============+
|           0.001 | 5.53971  |   0.14548  |
+-----------------+----------+------------+
|           0.003 | 0.884992 |   0.128023 |
+-----------------+----------+------------+
|           0.06  | 0.658802 |   0.139747 |
+-----------------+----------+------------+
|           0.01  | 0.659807 |   0.112322 |
+-----------------+----------+------------+
|           0.03  | 0.658852 |   0.115576 |
+-----------------+----------+------------+
|           0.06  | 0.658802 |   0.116462 |
+-----------------+----------+------------+
|           0.1   | 0.658801 |   0.11938  |
+-----------------+----------+------------+
Optimal learning rate: 0.1 with MSE: 0.6588010422671127 executed in time: 0.11937999725341797s


In [14]:
results = [] 
best_lr = None
best_mse = None
time_tracker = {}

print("Ridge SGD:")
for lr in learning_rates:
    start = time.time()
    model = RidgeRegressionSGD(learning_rate=lr, alpha=0.1, max_iter=1000)
    rsgd = model.fit(X_train, y_train)
    y_predicted = model.predict(X_test)
    mse = mean_squared_error(y_test, y_predicted)

    duration = time.time() - start
    time_tracker[lr] = duration
    
    results.append({"Learning Rate": lr, "MSE": mse, "Time (s)": duration})
    
    if best_lr is None or mse < best_mse:
        best_lr = lr
        best_mse = mse


print(tabulate(results, headers="keys", tablefmt="grid"))

print(f"Optimal learning rate: {best_lr} with MSE: {best_mse} executed in time: {time_tracker[best_lr]}s")



Ridge SGD:
+-----------------+----------+------------+
|   Learning Rate |      MSE |   Time (s) |
+=================+==========+============+
|           0.001 | 0.662536 |    30.7272 |
+-----------------+----------+------------+
|           0.003 | 0.68029  |    33.6914 |
+-----------------+----------+------------+
|           0.06  | 1.10565  |    29.5509 |
+-----------------+----------+------------+
|           0.01  | 0.70068  |    27.3548 |
+-----------------+----------+------------+
|           0.03  | 0.662874 |    30.0938 |
+-----------------+----------+------------+
|           0.06  | 0.960656 |    26.5846 |
+-----------------+----------+------------+
|           0.1   | 2.10127  |    27.8584 |
+-----------------+----------+------------+
Optimal learning rate: 0.001 with MSE: 0.6625358838304217 executed in time: 30.72719955444336s


In this example Ridge GD is faster and more efficient, with the optimal learning rate of 0.1 giving the best performance. Ridge SGD, while achieving a slightly lower MSE at 0.001, takes much longer to converge. This results are with alfa set to 0.1.

##### Checking with another value of alpha

In [17]:
from tabulate import tabulate
import time

# learning rates proposed in lecture
learning_rates = [0.001, 0.003,0.06, 0.01, 0.03, 0.06, 0.1]

results = [] 
best_lr = None
best_mse = None
time_tracker = {}

print("Ridge GD:")
for lr in learning_rates:
    start = time.time()
    model = RidgeRegressionGD(learning_rate=lr, alpha=0.01, max_iter=1000)
    rgd = model.fit(X_train, y_train)
    y_predicted = model.predict(X_test)
    mse = mean_squared_error(y_test, y_predicted)

    duration = time.time() - start
    time_tracker[lr] = duration
    
    results.append({"Learning Rate": lr, "MSE": mse, "Time (s)": duration})
    
    if best_lr is None or mse < best_mse:
        best_lr = lr
        best_mse = mse

# results as a table
print(tabulate(results, headers="keys", tablefmt="grid"))

print(f"Optimal learning rate: {best_lr} with MSE: {best_mse} executed in time: {time_tracker[best_lr]}s")



Ridge GD:
+-----------------+----------+------------+
|   Learning Rate |      MSE |   Time (s) |
+=================+==========+============+
|           0.001 | 4.93742  |   0.133456 |
+-----------------+----------+------------+
|           0.003 | 0.546468 |   0.132302 |
+-----------------+----------+------------+
|           0.06  | 0.50118  |   0.1304   |
+-----------------+----------+------------+
|           0.01  | 0.502345 |   0.125607 |
+-----------------+----------+------------+
|           0.03  | 0.501586 |   0.259258 |
+-----------------+----------+------------+
|           0.06  | 0.50118  |   0.227053 |
+-----------------+----------+------------+
|           0.1   | 0.501065 |   0.1788   |
+-----------------+----------+------------+
Optimal learning rate: 0.1 with MSE: 0.5010646863195825 executed in time: 0.17879986763000488s


In [18]:
results = [] 
best_lr = None
best_mse = None
time_tracker = {}

print("Ridge SGD:")
for lr in learning_rates:
    start = time.time()
    model = RidgeRegressionSGD(learning_rate=lr, alpha=0.01, max_iter=1000)
    rsgd = model.fit(X_train, y_train)
    y_predicted = model.predict(X_test)
    mse = mean_squared_error(y_test, y_predicted)

    duration = time.time() - start
    time_tracker[lr] = duration
    
    results.append({"Learning Rate": lr, "MSE": mse, "Time (s)": duration})
    
    if best_lr is None or mse < best_mse:
        best_lr = lr
        best_mse = mse


print(tabulate(results, headers="keys", tablefmt="grid"))

print(f"Optimal learning rate: {best_lr} with MSE: {best_mse} executed in time: {time_tracker[best_lr]}s")



Ridge SGD:
+-----------------+----------+------------+
|   Learning Rate |      MSE |   Time (s) |
+=================+==========+============+
|           0.001 | 0.49042  |    29.9824 |
+-----------------+----------+------------+
|           0.003 | 0.509216 |    32.0274 |
+-----------------+----------+------------+
|           0.06  | 1.22295  |  3636.25   |
+-----------------+----------+------------+
|           0.01  | 0.567312 |    32.0926 |
+-----------------+----------+------------+
|           0.03  | 0.640437 |    32.4871 |
+-----------------+----------+------------+
|           0.06  | 0.99322  |    28.199  |
+-----------------+----------+------------+
|           0.1   | 8.52473  |    28.5481 |
+-----------------+----------+------------+
Optimal learning rate: 0.001 with MSE: 0.49042003512825505 executed in time: 29.982351064682007s


When alpha is set to 0.01, Ridge GD has the optimal learning rate of 0.1 with the best performance at 0.501 MSE in time 0.179 seconds. On the other hand, Ridge SGD achieves a lower MSE of 0.490 now, with the optimal learning rate of 0.001 requiring almost 30 seconds for convergence. However, this alpha might lead to overfitting.